# Topic 18 — Unsupervised Learning: Clustering
### Theory → tiny example → from-scratch K-Means → sklearn (K-Means, Hierarchical, DBSCAN) → experiment.

We now leave labeled data behind. **Clustering** groups similar data points together based on
**similarity/distance** alone — no `y` involved. Useful for exploring structure in data you don't
have labels for yet, or for discovering natural groupings (e.g. types of toxic comments) before
you decide on a labeling scheme.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.datasets import make_blobs, make_moons
from scipy.cluster.hierarchy import dendrogram, linkage

rng = np.random.default_rng(0)

## 1. Similarity, distance, centroids

- **Distance**: same Euclidean distance from Topic 10 — how "close" two points are.
- **Centroid**: the average position (center) of a group of points.
- **Cluster assignment**: which group each point belongs to.

In [ ]:
X, true_labels = make_blobs(n_samples=150, centers=3, cluster_std=1.0, random_state=42)

plt.figure(figsize=(5, 4))
plt.scatter(X[:, 0], X[:, 1], alpha=0.7)
plt.title("Unlabeled data -- can you spot the groups?")
plt.show()
# A human can visually spot ~3 clusters here. Clustering algorithms do this automatically,
# and work in dimensions humans can't visualize (10s, 100s of features).

## 2. K-Means — from scratch

The algorithm, in a loop:
1. Pick `k` initial centroids (e.g. randomly from the data).
2. Assign every point to its NEAREST centroid.
3. Move each centroid to the average position of its assigned points.
4. Repeat steps 2-3 until assignments stop changing.

In [ ]:
def kmeans_scratch(X, k, n_iters=10):
    # Step 1: initialize centroids by picking k random points from the data
    centroid_idx = rng.choice(len(X), size=k, replace=False)
    centroids = X[centroid_idx].copy()

    for iteration in range(n_iters):
        # Step 2: assign each point to its nearest centroid
        distances = np.array([[np.linalg.norm(x - c) for c in centroids] for x in X])
        assignments = np.argmin(distances, axis=1)

        # Step 3: move each centroid to the mean of its assigned points
        new_centroids = np.array([
            X[assignments == i].mean(axis=0) if np.any(assignments == i) else centroids[i]
            for i in range(k)
        ])

        if np.allclose(new_centroids, centroids):
            print(f"converged after {iteration} iterations")
            break
        centroids = new_centroids

    return assignments, centroids

assignments, centroids = kmeans_scratch(X, k=3)

plt.figure(figsize=(5, 4))
plt.scatter(X[:, 0], X[:, 1], c=assignments, cmap="viridis", alpha=0.7)
plt.scatter(centroids[:, 0], centroids[:, 1], c="red", marker="X", s=200, label="centroids")
plt.legend()
plt.title("From-scratch K-Means result")
plt.show()

## 3. sklearn's KMeans, and choosing `k` with the elbow method

You must choose `k` in advance. The **elbow method**: run K-Means for several values of `k`, plot
`inertia_` (sum of squared distances to nearest centroid) vs `k`, and look for the "elbow" where
adding more clusters stops helping much.

In [ ]:
inertias = []
k_values = range(1, 8)
for k in k_values:
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(X)
    inertias.append(km.inertia_)

plt.figure(figsize=(5, 4))
plt.plot(k_values, inertias, marker="o")
plt.xlabel("k"); plt.ylabel("inertia")
plt.title("Elbow method for choosing k")
plt.show()
# Look for where the curve "bends" -- here it should bend around k=3, matching the true number of blobs.

In [ ]:
km = KMeans(n_clusters=3, n_init=10, random_state=42)
km_labels = km.fit_predict(X)

plt.figure(figsize=(5, 4))
plt.scatter(X[:, 0], X[:, 1], c=km_labels, cmap="viridis", alpha=0.7)
plt.scatter(km.cluster_centers_[:, 0], km.cluster_centers_[:, 1], c="red", marker="X", s=200)
plt.title("sklearn KMeans (k=3)")
plt.show()

## 4. Hierarchical clustering

Builds a tree of nested clusters (a **dendrogram**), either merging small clusters upward
(agglomerative — most common) or splitting one big cluster downward. Doesn't require choosing `k`
in advance — you can "cut" the tree at any height to get however many clusters you want.

In [ ]:
linked = linkage(X, method="ward")   # ward minimizes within-cluster variance at each merge

plt.figure(figsize=(8, 4))
dendrogram(linked, truncate_mode="lastp", p=15)
plt.title("Dendrogram (hierarchical clustering)")
plt.xlabel("sample index / cluster size")
plt.ylabel("distance")
plt.show()
# Cutting the tree at a chosen height determines how many clusters you end up with.

In [ ]:
agg = AgglomerativeClustering(n_clusters=3)
agg_labels = agg.fit_predict(X)

plt.figure(figsize=(5, 4))
plt.scatter(X[:, 0], X[:, 1], c=agg_labels, cmap="viridis", alpha=0.7)
plt.title("Agglomerative clustering (n_clusters=3)")
plt.show()

## 5. DBSCAN — density-based clustering

K-Means assumes round, similarly-sized clusters and struggles with weird shapes.
**DBSCAN** groups points that are densely packed together, and marks sparse points as **noise**
(no cluster at all) — it doesn't require choosing `k`, and handles arbitrary shapes.

In [ ]:
X_moons, _ = make_moons(n_samples=200, noise=0.08, random_state=42)

# K-Means struggles on this crescent shape (it assumes round clusters)
km_moons = KMeans(n_clusters=2, n_init=10, random_state=42).fit_predict(X_moons)

# DBSCAN handles it correctly
dbscan = DBSCAN(eps=0.2, min_samples=5)
db_labels = dbscan.fit_predict(X_moons)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].scatter(X_moons[:, 0], X_moons[:, 1], c=km_moons, cmap="bwr")
axes[0].set_title("K-Means (fails on crescent shapes)")
axes[1].scatter(X_moons[:, 0], X_moons[:, 1], c=db_labels, cmap="bwr")
axes[1].set_title("DBSCAN (correctly separates the two moons)")
plt.tight_layout()
plt.show()
print("DBSCAN found", len(set(db_labels)) - (1 if -1 in db_labels else 0), "clusters")
print("noise points (-1):", np.sum(db_labels == -1))

## Exercise

In [ ]:
# --- Try it yourself ---
# 1. Run kmeans_scratch with k=2 and k=5 on the original blob data X -- inspect the resulting plots.
# 2. Change DBSCAN's eps (try 0.1 and 0.5) on X_moons -- how does the number of noise points change?
# 3. Cut the dendrogram from part 4 at a different height (change `p` or inspect `linked` directly)
#    and describe how many clusters that would produce.
# 4. Where might clustering fit into YOUR cyberbullying research -- e.g. clustering toxic comments
#    by TF-IDF similarity (Topic 24) before you have labels, to explore what kinds of bullying
#    language exist in the dataset? Write 2-3 sentences.

---
### Next up: **Topic 19 — Dimensionality Reduction** (PCA, t-SNE, UMAP).

Say "next" when you're ready.